# Rekord Box Exploration

Make sure to create a backup before running scripts. 
The backup dialog can be found under "File" > "Library" > "Backup Library".

In [1]:
# automatically reload imported modules before executing code
%load_ext autoreload
%autoreload 2

In [15]:
from pyrekordbox import Rekordbox6Database
from pyrekordbox.db6 import tables
from typing import List

# Get table refs 
song_table = tables.DjmdContent
tag_table = tables.DjmdMyTag
song_tag_table = tables.DjmdSongMyTag

db = Rekordbox6Database()

tags: List[tables.DjmdMyTag] =  db.query(tag_table).all()

In [18]:
import polars as pl

schema = {
    "TagID": pl.Utf8,
    "Seq": pl.Int32,
    "Attribute": pl.Int32,
    "Name": pl.Utf8,
    "ParentID": pl.Utf8,
    "UUID": pl.Utf8,
}

raw_tags_df = pl.from_records(
    [{
        "TagID": t.ID,
        "Seq": t.Seq,
        "Attribute": t.Attribute,
        "Name": t.Name,
        "ParentID": t.ParentID,
        "UUID": t.UUID,
    } for t in tags
    ],
    schema=schema
)

In [19]:
raw_tags_df.head()

TagID,Seq,Attribute,Name,ParentID,UUID
str,i32,i32,str,str,str
"""1""",1,1,"""Genre""","""root""","""1"""
"""1429694612""",45,0,"""Techno""","""1""","""3e973c3c-001b-4792-92f9-684d40…"
"""2""",4,1,"""Years & Origin""","""root""","""2"""
"""3""",2,1,"""Situation""","""root""","""3"""
"""3506999954""",8,0,"""Peak""","""3""","""19bf7c0a-024e-4207-b088-80b5e5…"


In [22]:
root_groups_df = raw_tags_df.filter(
    pl.col("ParentID") == "root"
)

root_groups_df.head()

TagID,Seq,Attribute,Name,ParentID,UUID
str,i32,i32,str,str,str
"""1""",1,1,"""Genre""","""root""","""1"""
"""2""",4,1,"""Years & Origin""","""root""","""2"""
"""3""",2,1,"""Situation""","""root""","""3"""
"""4""",3,1,"""Mood""","""root""","""4"""


In [24]:
filtered_tags_df = raw_tags_df.filter(
    pl.col("ParentID") != "root"
)

filtered_tags_df.head()

TagID,Seq,Attribute,Name,ParentID,UUID
str,i32,i32,str,str,str
"""1429694612""",45,0,"""Techno""","""1""","""3e973c3c-001b-4792-92f9-684d40…"
"""3506999954""",8,0,"""Peak""","""3""","""19bf7c0a-024e-4207-b088-80b5e5…"
"""4144163695""",6,0,"""Loungy""","""3""","""5f66d396-e113-497c-a36a-94ed5d…"
"""3174622364""",7,0,"""Build up""","""3""","""37413d1c-39d4-4c51-b3c9-5e11d4…"
"""2484825285""",2,0,"""Ambient""","""1""","""22a59e2a-b3a6-470e-ad4b-92edda…"


In [25]:
filtered_tags_df.write_csv("../data/unique_tags.csv")